<a href="https://colab.research.google.com/github/scarletscherzinger/gnn-vs-mlp-benchmark/blob/main/benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 0. Setup and Install Dependencies
# ==========================================
import os
import torch
os.environ['TORCH'] = torch.__version__

print("Checking PyTorch and PyTorch Geometric installation...")
try:
    import torch_geometric
    print(f"PyG version: {torch_geometric.__version__}")
except ImportError:
    print("Installing PyTorch Geometric...")
    !pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-${TORCH}.html
    import torch_geometric

import time
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, SAGEConv

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==========================================
# 1. Load Cora Dataset
# ==========================================
dataset = Planetoid(root='./data/Cora', name='Cora')
data = dataset[0].to(device)
print(f"Dataset loaded: {dataset}")
print(f"Number of nodes: {data.num_nodes}, Number of edges: {data.num_edges}")

# ==========================================
# 2. Define Models (MLP, GCN, GraphSAGE)
# ==========================================
class MLP(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin2(x)
        return x

class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# ==========================================
# 3. Training & Evaluation Helper Function
# ==========================================
def train_and_evaluate(model_class, seeds, epochs=200):
    accuracies = []

    for seed in seeds:
        # Set seeds for reproducibility
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        model = model_class(dataset.num_features, 16, dataset.num_classes).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

        # Training Loop
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            out = model(data.x, data.edge_index)
            loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()

        # Evaluation Loop
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=-1)
            correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
            acc = int(correct) / int(data.test_mask.sum())
            accuracies.append(acc * 100) # Convert to percentage

    return accuracies

# ==========================================
# 4. Systems Metrics (Parameters & Latency)
# ==========================================
def measure_systems_metrics(model_class):
    model = model_class(dataset.num_features, 16, dataset.num_classes).to(device)
    model.eval()

    # Parameter Count
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Inference Latency (Average over 100 runs)
    iterations = 100
    with torch.no_grad():
        # Warmup
        for _ in range(10):
            _ = model(data.x, data.edge_index)

        start_time = time.time()
        for _ in range(iterations):
            _ = model(data.x, data.edge_index)
        end_time = time.time()

    avg_inference_time_ms = ((end_time - start_time) / iterations) * 1000
    return num_params, avg_inference_time_ms

# ==========================================
# 5. Run Experiment Across 10 Seeds
# ==========================================
SEEDS = [42, 100, 7, 99, 123, 456, 789, 321, 654, 987]
models_dict = {'MLP (No Graph)': MLP, 'GCN': GCN, 'GraphSAGE': GraphSAGE}

results = []

print("\nRunning experiments across 10 random seeds...")
for name, model_cls in models_dict.items():
    print(f"-> Training {name}...")
    accs = train_and_evaluate(model_cls, SEEDS)
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)

    params, latency = measure_systems_metrics(model_cls)

    results.append({
        'Model': name,
        'Accuracy (%)': f"{mean_acc:.2f} ± {std_acc:.2f}",
        'Parameters': params,
        'Inference Time (ms)': f"{latency:.4f}"
    })

# ==========================================
# 6. Display Final Results Table
# ==========================================
print("\n" + "="*70)
print(f"{'Model':<20} | {'Test Accuracy (%)':<20} | {'Parameters':<12} | {'Latency (ms)':<12}")
print("="*70)
for r in results:
    print(f"{r['Model']:<20} | {r['Accuracy (%)']:<20} | {r['Parameters']:<12} | {r['Inference Time (ms)']:<12}")
print("="*70)

Checking PyTorch and PyTorch Geometric installation...
Installing PyTorch Geometric...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.1/682.1 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.0/828.0 kB 918.5 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.2 MB/s eta 0:00:00
Using device: cpu


Processing...
Done!


Dataset loaded: Cora()
Number of nodes: 2708, Number of edges: 10556

Running experiments across 10 random seeds...
-> Training MLP (No Graph)...
-> Training GCN...
-> Training GraphSAGE...

Model                | Test Accuracy (%)    | Parameters   | Latency (ms)
MLP (No Graph)       | 51.88 ± 3.16         | 23063        | 3.0338      
GCN                  | 80.27 ± 0.65         | 23063        | 9.6502      
GraphSAGE            | 79.44 ± 0.87         | 46103        | 60.1546     
